# At what granularity does your graph stop being degenerate?

**Question this notebook answers:** Shell commands can be parsed at many levels
of detail. Too coarse — every command maps to its bare binary (`grep`, `curl`)
— and all commands look the same: the graph becomes a near-clique where every
node connects to nearly every other. Too fine — each URL or file path is its
own node — and the graph shatters into dust: most nodes are singletons with no
neighbours.

This notebook sweeps several granularity settings and shows you where the
**informative band** is, if it exists at all.

## The two degenerate regimes

| Regime | What it looks like | Why it is useless |
|---|---|---|
| **Too dense** | density > 15%, almost every node connects to every other | Node A's neighbourhood ≈ node B's neighbourhood — no discrimination |
| **Too sparse** | density < 0.2%, most nodes have degree 0 | Nothing to look at — no co-occurrence signal |

The informative band observed on real traces: **density 1–6%, mean degree
10–15**. These numbers are empirical — measure your own.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
SESSIONS_GLOB = "~/.openclaw/agents/*/sessions/*.jsonl"

# Optional: filter to a specific agent name (leave empty to include all).
AGENT_FILTER = ""
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import glob
import os

from agentgraph import build_hyperedges, metrics, granularity_sweep

matched = glob.glob(os.path.expanduser(SESSIONS_GLOB))
if not matched:
    raise FileNotFoundError(
        f"No session files found for pattern: {SESSIONS_GLOB!r}\n"
        "Update SESSIONS_GLOB in the configuration cell above."
    )
print(f"{len(matched)} session file(s) found.")

In [ ]:
# Default granularity: a good starting point for most agent traces.
# url_segments=3 keeps host + first two URL path components.
# path_components=2 keeps the first two components of file paths.

edges = build_hyperedges(SESSIONS_GLOB)

if AGENT_FILTER:
    edges = [e for e in edges if e.agent == AGENT_FILTER]

print(f"Hyperedges (default granularity): {len(edges)}")
print()
print(metrics(edges))

In [ ]:
# Granularity sweep.
# Each variant changes how finely URLs and file paths are resolved.
# The goal is to find the setting where density sits in the 1–6% band.

VARIANTS = {
    # Coarsest: bare binary only, no resource info at all.
    "binary-only (url=0, path=1)": {"url_segments": 0, "path_components": 1},
    # Medium-coarse: host + one URL segment, first path component.
    "coarse       (url=1, path=1)": {"url_segments": 1, "path_components": 1},
    # DEFAULT: host + two segments, two path components.
    "default      (url=3, path=2)": {"url_segments": 3, "path_components": 2},
    # Fine: host + three URL segments, three path components.
    "fine         (url=3, path=3)": {"url_segments": 3, "path_components": 3},
    # Finest: maximum detail — likely too sparse.
    "very fine    (url=5, path=4)": {"url_segments": 5, "path_components": 4},
}

rows = granularity_sweep(SESSIONS_GLOB, VARIANTS)

# Filter rows if an agent filter is set.
# (Note: granularity_sweep builds from scratch each time; agent filtering
# is best applied by restricting the glob to a single agent's directory.)

print(f"{'Variant':<36} {'edges':>6} {'N':>5} {'density':>8} {'mean_deg':>9} {'entropy':>8} {'verdict'}")
print("-" * 90)
for row in rows:
    print(
        f"{row['variant']:<36} "
        f"{row['hyperedges']:>6} "
        f"{row['N']:>5} "
        f"{row['density']:>8.4f} "
        f"{row['mean_degree']:>9.1f} "
        f"{row['entropy']:>8.3f}  "
        f"{row['verdict']}"
    )

In [ ]:
# Concrete demonstration of the two degenerate regimes.
# This cell shows what the graph looks like at both extremes so you can
# recognise them in your own sweep table above.

print("=" * 60)
print("DEGENERATE REGIME 1: binary-only (likely TOO DENSE)")
print("=" * 60)
edges_coarse = build_hyperedges(SESSIONS_GLOB, url_segments=0, path_components=1)
if AGENT_FILTER:
    edges_coarse = [e for e in edges_coarse if e.agent == AGENT_FILTER]
print(metrics(edges_coarse))

print()
print("=" * 60)
print("DEGENERATE REGIME 2: very fine (likely TOO SPARSE)")
print("=" * 60)
edges_fine = build_hyperedges(SESSIONS_GLOB, url_segments=5, path_components=4)
if AGENT_FILTER:
    edges_fine = [e for e in edges_fine if e.agent == AGENT_FILTER]
print(metrics(edges_fine))

## How to read the sweep table

Look for the row where **density is between 0.01 and 0.06** and **mean degree
is between 10 and 15**. That is your working granularity — use it in
notebooks 03 and 04.

**If no row falls in the informative band:**

- All rows show TOO DENSE → your vocabulary is too narrow. Your agent uses
  very few distinct tools. The graph cannot discriminate between actions.
- All rows show TOO SPARSE → your corpus is too small, or your agent uses
  a very large number of unique commands. Collect more traces.
- A single row sits in the band → that is your answer. Note the settings
  (`url_segments`, `path_components`) and use them in notebooks 03 and 04
  by passing them as keyword arguments to `build_hyperedges()`.

The informative band is not guaranteed to exist for every agent. This is a
finding in itself — it means the agent's actions do not form a reusable
vocabulary at any level of detail.